In [ ]:
!pip -q install scikit-learn openpyxl joblib

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

np.random.seed(42)

MODEL_NAME = "SVM_TFIDF_MULTILABEL_WITH_EMOJI"

TRAIN_PATH = "train.xlsx"
VALID_PATH = "valid.xlsx"
TEST_PATH  = "test.xlsx"

TEXT_COL = "clean_text"
LABEL_DIALECT_COL = "dialect"
LABEL_HATE_COL    = "hate"
LABEL_TOPIC_COL   = "topic"

OUTPUT_DIR = "./svm_multilabel_model"
MODEL_PATH = "svm_multilabel_model.joblibfinalone"

id2dialect = {1: "Saudi", 0: "Egyptian"}
id2hate    = {1: "Hate", 0: "Not Hate"}
id2topic   = {1: "Religious", 0: "Political"}

HATE_EMOJIS = set([
    '💦', '🐖', '🐷', '🐽', '👞', '🐕', '🐶', '💩', '🐄', '🐮',
    '🐑', '🐏', '👎', '😡', '🤬', '👺', '👿', '😠'
])

def emoji_flag(text: str) -> int:
    return 1 if any(e in str(text) for e in HATE_EMOJIS) else 0

def load_excel(path: str) -> pd.DataFrame:
    return pd.read_excel(path, engine="openpyxl")

def load_and_validate_xlsx(path: str) -> pd.DataFrame:
    df = load_excel(path)

    required = {TEXT_COL, LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{path} missing columns: {missing}\n"
            f"Available columns: {list(df.columns)}"
        )

    df = df.copy()
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str).str.strip()

    for c in [LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL]:
        if df[c].isna().any():
            bad_rows = df[df[c].isna()].head(5)
            raise ValueError(
                f"{path}: column '{c}' has NaN values. Examples:\n"
                f"{bad_rows[[TEXT_COL, c]]}"
            )

        df[c] = pd.to_numeric(df[c], errors="raise").astype(int)

        bad = ~df[c].isin([0, 1])
        if bad.any():
            examples = df.loc[bad, [TEXT_COL, c]].head(5)
            raise ValueError(
                f"{path}: column '{c}' must be 0/1 only. Examples:\n"
                f"{examples}"
            )

    return df

train_df = load_and_validate_xlsx(TRAIN_PATH)
valid_df = load_and_validate_xlsx(VALID_PATH)
test_df  = load_and_validate_xlsx(TEST_PATH)

print("Rows:", len(train_df), len(valid_df), len(test_df))
print("Train label counts:")
print("dialect:", train_df[LABEL_DIALECT_COL].value_counts().to_dict())
print("hate:",    train_df[LABEL_HATE_COL].value_counts().to_dict())
print("topic:",   train_df[LABEL_TOPIC_COL].value_counts().to_dict())

class MultiLabelSVMDataset:
    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def get_x(self):
        x = self.df[[TEXT_COL]].copy()
        x[TEXT_COL] = x[TEXT_COL].astype(str)
        return x

    def get_labels(self):
        return self.df[
            [LABEL_DIALECT_COL, LABEL_HATE_COL, LABEL_TOPIC_COL]
        ].values.astype(int)

train_ds = MultiLabelSVMDataset(train_df)
valid_ds = MultiLabelSVMDataset(valid_df)
test_ds  = MultiLabelSVMDataset(test_df)

class EmojiFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.hate_emojis = HATE_EMOJIS

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([
            [1 if any(e in str(text) for e in self.hate_emojis) else 0]
            for text in X
        ])

class MultiLabelSVM:
    def __init__(self):
        self.model = Pipeline([
            ("features", ColumnTransformer([
                (
                    "text_tfidf",
                    TfidfVectorizer(
                        analyzer="char_wb",
                        ngram_range=(3, 5),
                        max_features=50000,
                        min_df=2
                    ),
                    TEXT_COL
                ),
                (
                    "emoji_feature",
                    EmojiFeatureExtractor(),
                    TEXT_COL
                ),
            ])),
            (
                "classifier",
                OneVsRestClassifier(
                    LinearSVC(
                        C=1.0,
                        class_weight="balanced",
                        random_state=42
                    )
                )
            )
        ])

    def fit(self, train_dataset):
        X_train = train_dataset.get_x()
        y_train = train_dataset.get_labels()
        self.model.fit(X_train, y_train)

    def predict(self, dataset):
        X = dataset.get_x()
        preds = self.model.predict(X)
        return np.asarray(preds).astype(int)

    def decision_function_text(self, text: str):
        df = pd.DataFrame({TEXT_COL: [str(text)]})
        scores = self.model.decision_function(df)
        scores = np.asarray(scores)
        if scores.ndim == 1:
            scores = scores.reshape(1, -1)
        return scores[0]

    def save(self, path: str):
        joblib.dump(self.model, path)

model = MultiLabelSVM()

def compute_metrics(labels, preds):
    labels = np.asarray(labels)
    preds = np.asarray(preds)

    metrics = {}
    metrics["overall_acc"] = np.mean(np.all(labels == preds, axis=1))

    metrics["dialect_acc"] = accuracy_score(labels[:, 0], preds[:, 0])
    metrics["dialect_f1"] = f1_score(labels[:, 0], preds[:, 0], average="macro", zero_division=0)
    metrics["dialect_f1_micro"] = f1_score(labels[:, 0], preds[:, 0], average="micro", zero_division=0)

    metrics["hate_acc"] = accuracy_score(labels[:, 1], preds[:, 1])
    metrics["hate_f1"] = f1_score(labels[:, 1], preds[:, 1], average="macro", zero_division=0)
    metrics["hate_f1_micro"] = f1_score(labels[:, 1], preds[:, 1], average="micro", zero_division=0)

    metrics["topic_acc"] = accuracy_score(labels[:, 2], preds[:, 2])
    metrics["topic_f1"] = f1_score(labels[:, 2], preds[:, 2], average="macro", zero_division=0)
    metrics["topic_f1_micro"] = f1_score(labels[:, 2], preds[:, 2], average="micro", zero_division=0)

    metrics["avg_f1"] = (
        metrics["dialect_f1"] +
        metrics["hate_f1"] +
        metrics["topic_f1"]
    ) / 3.0

    metrics["avg_f1_micro"] = (
        metrics["dialect_f1_micro"] +
        metrics["hate_f1_micro"] +
        metrics["topic_f1_micro"]
    ) / 3.0

    return metrics

print("\nTraining SVM model...")
model.fit(train_ds)
print("Training done.")

valid_labels = valid_ds.get_labels()
valid_preds = model.predict(valid_ds)
valid_metrics = compute_metrics(valid_labels, valid_preds)

print("\n========== Validation Metrics ==========")
for k, v in valid_metrics.items():
    print(f"{k}: {v:.4f}")

labels = test_ds.get_labels()
preds = model.predict(test_ds)
test_metrics = compute_metrics(labels, preds)

print("\n========== Test Metrics ==========")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

overall_acc = np.mean(np.all(labels == preds, axis=1))
print(f"\nOverall exact-match accuracy (all labels together): {overall_acc:.4f}")

def print_report_with_micro(y_true, y_pred, title: str):
    rep = classification_report(
        y_true,
        y_pred,
        digits=4,
        zero_division=0,
        output_dict=True
    )

    f1_micro = f1_score(
        y_true,
        y_pred,
        average="micro",
        zero_division=0
    )

    acc = accuracy_score(y_true, y_pred)

    def row(k):
        return rep.get(k, {
            "precision": 0.0,
            "recall": 0.0,
            "f1-score": 0.0,
            "support": 0
        })

    r0 = row("0")
    r1 = row("1")
    rmacro = row("macro avg")
    rweight = row("weighted avg")
    total_support = int(r0["support"]) + int(r1["support"])

    print(f"\n--- {title} ---")
    print(f"{'':>14}{'precision':>10}{'recall':>10}{'f1-score':>10}{'f1-micro':>10}{'support':>10}")
    print(f"{'0.0':>14}{r0['precision']:>10.4f}{r0['recall']:>10.4f}{r0['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r0['support']):>10}")
    print(f"{'1.0':>14}{r1['precision']:>10.4f}{r1['recall']:>10.4f}{r1['f1-score']:>10.4f}{f1_micro:>10.4f}{int(r1['support']):>10}")
    print()
    print(f"{'accuracy':>14}{'':>10}{'':>10}{acc:>10.4f}{f1_micro:>10.4f}{total_support:>10}")
    print(f"{'macro avg':>14}{rmacro['precision']:>10.4f}{rmacro['recall']:>10.4f}{rmacro['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rmacro['support']):>10}")
    print(f"{'weighted avg':>14}{rweight['precision']:>10.4f}{rweight['recall']:>10.4f}{rweight['f1-score']:>10.4f}{f1_micro:>10.4f}{int(rweight['support']):>10}")

print_report_with_micro(
    labels[:, 0].astype(int),
    preds[:, 0].astype(int),
    "Dialect report (1=saudi,0=egyptian)"
)

print_report_with_micro(
    labels[:, 1].astype(int),
    preds[:, 1].astype(int),
    "Hate report (1=hate,0=not hate)"
)

print_report_with_micro(
    labels[:, 2].astype(int),
    preds[:, 2].astype(int),
    "Topic report (1=religious,0=political)"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
model.save(MODEL_PATH)

print("\nSaved to:", MODEL_PATH)

out_df = test_df.copy()

out_df["pred_dialect"] = preds[:, 0]
out_df["pred_hate"] = preds[:, 1]
out_df["pred_topic"] = preds[:, 2]

out_df["pred_dialect_label"] = out_df["pred_dialect"].map(id2dialect)
out_df["pred_hate_label"] = out_df["pred_hate"].map(id2hate)
out_df["pred_topic_label"] = out_df["pred_topic"].map(id2topic)

out_path = os.path.join(OUTPUT_DIR, "svm_test_predictions.xlsx")
out_df.to_excel(out_path, index=False)

print("Saved predictions to:", out_path)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def predict(text: str, threshold: float = 0.5):
    scores = model.decision_function_text(text)
    probs = sigmoid(scores)
    pred = (probs >= threshold).astype(int)

    dialect_id = int(pred[0])
    hate_id = int(pred[1])
    topic_id = int(pred[2])

    dialect_p = float(probs[0])
    hate_p = float(probs[1])
    topic_p = float(probs[2])

    return {
        "dialect_id": dialect_id,
        "hate_id": hate_id,
        "topic_id": topic_id,
        "dialect_label": id2dialect[dialect_id],
        "hate_label": id2hate[hate_id],
        "topic_label": id2topic[topic_id],
        "dialect_probs": {"Egyptian": 1 - dialect_p, "Saudi": dialect_p},
        "hate_probs": {"Not Hate": 1 - hate_p, "Hate": hate_p},
        "topic_probs": {"Political": 1 - topic_p, "Religious": topic_p},
        "dialect_confidence": dialect_p if dialect_id == 1 else (1 - dialect_p),
        "hate_confidence": hate_p if hate_id == 1 else (1 - hate_p),
        "topic_confidence": topic_p if topic_id == 1 else (1 - topic_p),
        "emoji_flag": int(emoji_flag(text)),
    }

print("\n=== Interactive Mode ===")
print("Type: exit / quit / stop to end\n")

while True:
    text = input("Tweet> ").strip()

    if text.lower() in {"exit", "quit", "stop"}:
        print("Done.")
        break

    if not text:
        continue

    result = predict(text, threshold=0.5)

    print(
        f"Dialect: {result['dialect_label']} "
        f"(confidence={result['dialect_confidence']:.3f}, "
        f"Egyptian={result['dialect_probs']['Egyptian']:.3f}, "
        f"Saudi={result['dialect_probs']['Saudi']:.3f})"
    )

    print(
        f"Hate: {result['hate_label']} "
        f"(confidence={result['hate_confidence']:.3f}, "
        f"Not Hate={result['hate_probs']['Not Hate']:.3f}, "
        f"Hate={result['hate_probs']['Hate']:.3f}, "
        f"emoji_flag={result['emoji_flag']})"
    )

    print(
        f"Topic: {result['topic_label']} "
        f"(confidence={result['topic_confidence']:.3f}, "
        f"Political={result['topic_probs']['Political']:.3f}, "
        f"Religious={result['topic_probs']['Religious']:.3f})"
    )

Rows: 2773 595 594
Train label counts:
dialect: {1: 1417, 0: 1356}
hate: {1: 1443, 0: 1330}
topic: {0: 1407, 1: 1366}

Training SVM model...
Training done.

========== Validation Metrics ==========
overall_acc: 0.6891
dialect_acc: 0.8908
dialect_f1: 0.8907
dialect_f1_micro: 0.8908
hate_acc: 0.7933
hate_f1: 0.7933
hate_f1_micro: 0.7933
topic_acc: 0.9815
topic_f1: 0.9815
topic_f1_micro: 0.9815
avg_f1: 0.8885
avg_f1_micro: 0.8885

========== Test Metrics ==========
overall_acc: 0.6818
dialect_acc: 0.8990
dialect_f1: 0.8989
dialect_f1_micro: 0.8990
hate_acc: 0.7660
hate_f1: 0.7660
hate_f1_micro: 0.7660
topic_acc: 0.9832
topic_f1: 0.9831
topic_f1_micro: 0.9832
avg_f1: 0.8827
avg_f1_micro: 0.8827

Overall exact-match accuracy (all labels together): 0.6818

--- Dialect report (1=saudi,0=egyptian) ---
               precision    recall  f1-score  f1-micro   support
           0.0    0.9358    0.8711    0.9023    0.8990       318
           1.0    0.8624    0.9312    0.8955    0.8990       276
